# Neural Latent Representation Analysis

Explore the learned neural representations: extract embeddings, visualize with
PCA and t-SNE, plot neural trajectories through latent space, compute electrode
importance via gradient-based saliency, and build trial similarity matrices.

## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
%matplotlib inline

from src.config import get_default_config
from src.data.loader import load_willett_dataset
from src.data.dataset import create_dataloaders
from src.models.gru_decoder import GRUDecoder
from src.analysis.embeddings import extract_embeddings
from src.analysis.trajectory_plots import plot_neural_trajectory
from src.analysis.saliency import compute_saliency
from src.analysis.similarity_matrix import compute_similarity_matrix
from src.visualization.embedding_plots import plot_embedding_scatter

## 2. Load Model and Data

In [ ]:
cfg = get_default_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

trial_index = load_willett_dataset(cfg)
_, val_loader = create_dataloaders(trial_index, cfg)

# Load GRU model
model = GRUDecoder(
    input_size=cfg.n_channels,
    hidden_size=cfg.hidden_size,
    num_layers=cfg.num_layers,
    num_classes=cfg.num_classes,
).to(device)

weight_path = os.path.join('..', 'checkpoints', 'gru_decoder_best.pt')
if os.path.exists(weight_path):
    model.load_state_dict(torch.load(weight_path, map_location=device))
    print(f"Loaded weights from {weight_path}")
else:
    print("Using randomly initialized weights (no checkpoint found)")

model.eval()
print(f"Model ready on {device}")

## 3. Extract Embeddings

Extract hidden-state embeddings from the model for all validation trials.

In [ ]:
embeddings, labels = extract_embeddings(model, val_loader, device=device)
print(f"Embeddings shape: {embeddings.shape}")
print(f"Labels count: {len(labels)}")

## 4. PCA Visualization

Project embeddings to 2D using PCA and color by character label.

In [ ]:
pca = PCA(n_components=2)
emb_pca = pca.fit_transform(embeddings)

fig = plot_embedding_scatter(
    emb_pca, labels,
    title='PCA Embedding Space',
    method='pca'
)
plt.show()

print(f"Explained variance: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")

## 5. t-SNE Visualization

Use t-SNE for non-linear dimensionality reduction to reveal cluster structure.

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
emb_tsne = tsne.fit_transform(embeddings)

fig = plot_embedding_scatter(
    emb_tsne, labels,
    title='t-SNE Embedding Space',
    method='tsne'
)
plt.show()

## 6. Neural Trajectories

Visualize how neural activity evolves over time in a low-dimensional space
for individual trials. This reveals the temporal dynamics of decoding.

In [ ]:
# Extract per-timestep embeddings for a few sample trials
sample_batch = next(iter(val_loader))
sample_signals = sample_batch['signals'][:3].to(device)

for i in range(sample_signals.shape[0]):
    fig = plot_neural_trajectory(
        model, sample_signals[i:i+1],
        title=f'Neural Trajectory — Trial {i+1}',
        device=device
    )
    plt.show()

## 7. Electrode Saliency / Importance

Compute gradient-based saliency to identify which electrodes contribute most
to the model's decoding decisions.

In [ ]:
sample_signal = sample_batch['signals'][0:1].to(device)
saliency = compute_saliency(model, sample_signal, device=device)

# Aggregate across time to get per-channel importance
channel_importance = saliency.mean(axis=0) if saliency.ndim > 1 else saliency

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(channel_importance)), channel_importance, color='coral', alpha=0.8)
ax.set_xlabel('Channel Index')
ax.set_ylabel('Saliency (Mean |Gradient|)')
ax.set_title('Electrode Importance via Gradient Saliency')
plt.tight_layout()
plt.show()

# Top 10 most important channels
top_k = 10
top_channels = np.argsort(channel_importance)[-top_k:][::-1]
print(f"Top {top_k} most important channels: {top_channels}")
for ch in top_channels:
    print(f"  Channel {ch:3d}: saliency = {channel_importance[ch]:.6f}")

## 8. Trial Similarity Matrix

Compute cosine similarity between trial embeddings to see which characters
and words are represented similarly in the model's latent space.

In [ ]:
sim_matrix = compute_similarity_matrix(embeddings)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='viridis', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_xlabel('Trial Index')
ax.set_ylabel('Trial Index')
ax.set_title('Trial Similarity Matrix (Embedding Space)')
plt.tight_layout()
plt.show()

# Print summary
upper_tri = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]
print(f"Mean pairwise similarity: {np.mean(upper_tri):.4f}")
print(f"Std  pairwise similarity: {np.std(upper_tri):.4f}")

---

**Next:** Proceed to `06_demo_visualization.ipynb` for a full end-to-end decoding walkthrough.